# Some plots for Analyzing the SQGp1 Data

In [ ]:
# Packages

import jax
# Enable 64-bit
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import jit
import jaxopt
import matplotlib
# matplotlib.use('Agg')          # headless, no Tk
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import tkinter as tk
from tkinter import filedialog
import scipy.io as sio
import sys
import dill
import inspect

# For fourier transform
from numpy.fft import rfft2, irfft2

# Input subfiles
%reload_ext autoreload
%autoreload 2

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), '..'))
from physics_functions import calculate_surface_u, forward_ssh
from cost_functions import cost_function_fmin
from ssh_setup import ssh_setup

import sys

print("Select sqgp1_for_thomas.mat ... Cancel to use the default path.")
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
file_path = filedialog.askopenfilename(
    title="Select an SQG+1 .mat file (with psi0n, p1n, Ro)",
    filetypes=[("MATLAB Data Files", "*.mat"), ("All Files", "*.*")],
)
root.destroy()

if not file_path:
    file_path = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                             "data", "sqgp1_for_thomas.mat")
    print(f"No file selected, using default: {file_path}")

mat_contents = sio.loadmat(file_path)
print(f"Loaded: {file_path}")
print(f"Keys:   {[k for k in mat_contents.keys() if not k.startswith('__')]}")

# Required fields
psi0n = jnp.asarray(mat_contents["psi0n"], dtype=jnp.float64)
p1n   = jnp.asarray(mat_contents["p1n"],   dtype=jnp.float64)
Ro_mat = float(mat_contents["Ro"].squeeze())

# Override the system parameters with the value used in the simulation
Ro_sys  = Ro_mat
epsilon = Ro_sys
print(f"  Ro from file -> Ro_sys = epsilon = {Ro_sys}")

Nx, Ny = psi0n.shape
print(f"Grid size adjusted automatically to Nx={Nx}, Ny={Ny}.")
data_name = os.path.splitext(os.path.basename(file_path))[0]

# Auxiliary fields — kept for diagnostics / plotting
b_s    = jnp.asarray(mat_contents["bn"],     dtype=jnp.float64)
zeta0n = jnp.asarray(mat_contents["zeta0n"], dtype=jnp.float64)
zeta1n = jnp.asarray(mat_contents["zeta1n"], dtype=jnp.float64)
deltan = jnp.asarray(mat_contents["deltan"], dtype=jnp.float64)



# Grid Setup

In [ ]:
# Grid Setup
Lx = 2.0 * jnp.pi
Ly = 2.0 * jnp.pi
Bu = 1.0
dx = Lx / Nx
dy = Ly / Ny
x = jnp.arange(Nx) * dx
y = jnp.arange(Ny) * dy
X, Y = jnp.meshgrid(x, y, indexing='ij')

# Spectral grid
dk = 1.0
dl = 1.0
k_zonal     = jnp.concatenate([jnp.arange(Nx//2), jnp.arange(-Nx//2, 0)]) * dk
l_meridional = jnp.concatenate([jnp.arange(Ny//2), jnp.arange(-Ny//2, 0)]) * dl
k_zonal      = k_zonal.at[Nx//2].set(0.0)      # kill Nyquist
l_meridional = l_meridional.at[Ny//2].set(0.0)
kx, ky = jnp.meshgrid(k_zonal, l_meridional, indexing='ij')

# The square of K
K2 = kx**2 + ky**2
K = jnp.sqrt(K2)

# Safe inverses (avoid division by zero at k=0)
inv_K = jnp.where(K > 0, 1.0 / K, 0.0)
inv_K2 = jnp.where(K2 > 0, 1.0 / K2, 0.0)

# This is for taking derivative in $z$ direction.
mu = jnp.sqrt(Bu) * K
inv_mu = jnp.where(mu > 0, 1.0 / mu, 0.0)


# Vorticity

First and Zero-th order vorticity

In [ ]:
# ── Surface vorticity from the .mat data (zeta0n, zeta1n, sum, diff) ──
# Each panel uses its own pct-th percentile of |field| as the symmetric
# vmax, so a few extreme cells don't squash all the small-scale structure.

zeta_sum  = zeta0n + Ro_sys * zeta1n
zeta_diff = zeta0n - Ro_sys * zeta1n

fields = [
    (zeta0n,    r"$\zeta_0$  (zeroth-order, QG)"),
    (Ro_sys * zeta1n,    r"$\zeta_1$  (first-order correction)"),
    (zeta_sum,  r"$\zeta_0 + \text{Ro} \cdot \zeta_1$"),
    (zeta_diff, r"$\zeta_0 - \text{Ro} \cdot \zeta_1$"),
]

extent = [0, float(Lx), 0, float(Ly)]
pct = 99   # color-saturation percentile

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)

for ax, (field, title) in zip(axes.ravel(), fields):
    vmax = float(jnp.percentile(jnp.abs(field), pct))
    im = ax.imshow(field, origin='lower', cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, extent=extent)
    ax.set_title(f"{title}\n", fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    plt.colorbar(im, ax=ax, fraction=0.046, label=r"$\zeta$")

plt.show()

print(f"  rms zeta0   = {float(jnp.sqrt(jnp.mean(zeta0n**2))):.4e}")
print(f"  rms zeta1   = {float(jnp.sqrt(jnp.mean(zeta1n**2))):.4e}")
print(f"  rms sum     = {float(jnp.sqrt(jnp.mean(zeta_sum**2))):.4e}")
print(f"  rms diff    = {float(jnp.sqrt(jnp.mean(zeta_diff**2))):.4e}")
